In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("detr")

import detr.util.misc as utils
from detr.engine import evaluate, train_one_epoch
from detr.models import build_model

import yaml
from types import SimpleNamespace
from PIL import Image

import os
import math
import glob
import tqdm
import matplotlib.pyplot as plt

import wandb

import torch
import torchvision.transforms.v2 as transforms
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader


from lib.dataset import FashionDataset
from lib.loss import Loss
from lib.utils import create_transforms
from lib.detr_test_utils import detr_collate_fn, train

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
use_imagenet_norm = False
train_transforms = create_transforms("train", use_imagenet_norm)
test_transforms = create_transforms("test", use_imagenet_norm)

train_dataset = FashionDataset("train", "../data/train/annotations", "../data/train/images", train_transforms)
test_dataset = FashionDataset("test", "../data/test/annotations", "../data/test/images", test_transforms)

MAX_OBJS = 5
max_classes = 13 + 1

with open("detr/default_params.yaml", 'r') as f:
    args = yaml.load(f, yaml.SafeLoader)
args["num_queries"] = MAX_OBJS
args["num_classes"] = max_classes
args["device"] = "mps"
args = SimpleNamespace(**args)

net, loss_fn, postprocessors = build_model(args)

batch_size = 8
num_workers = 0
train_dataloader = DataLoader(train_dataset, batch_size, shuffle=True, num_workers=num_workers, collate_fn=detr_collate_fn)
test_dataloader = DataLoader(test_dataset, batch_size, shuffle=False, num_workers=num_workers, collate_fn=detr_collate_fn)

dataloaders = {
    "train": train_dataloader,
    "test": test_dataloader
}

lr = 1e-4
momentum = 0.9

use_pretrained = False
if use_pretrained:
    optimizer = torch.optim.Adam([
            {"params": net.backbone.parameters(), "lr": 1e-5},
            {"params": net.decoder.parameters(), "lr": 2e-4},
            {"params": net.queries, "lr": 2e-4},
            {"params": net.category.parameters(), "lr": 2e-4},
            {"params": net.boxes.parameters(), "lr": 2e-4},
        ])
else:
    optimizer = torch.optim.Adam(net.parameters(), lr)
    
device = torch.device(args.device)

/Users/lorenzo/.pyenv/versions/3.10.9/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/lorenzo/.pyenv/versions/3.10.9/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [9]:
net.to(device)

n_epochs = 7
use_wandb = False
try:
    loss_results = train(net, loss_fn, optimizer, dataloaders, device, n_epochs, use_wandb)
finally:
    if use_wandb:
        wandb.finish()

Epoch 1/7


  1%|          | 21/2282 [00:25<45:41,  1.21s/it] 


KeyboardInterrupt: 